# ASX Historic Connectivity and EDA

Validation notebook for the ASX historic tabular overlay.

This notebook checks source URL accessibility, inspects MinIO-backed raw/conformed/curated artifacts, and performs lightweight EDA on the landed dataset.


In [ ]:
from __future__ import annotations

import io
import json
import os
from pathlib import Path
from urllib.parse import urlparse
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import pandas as pd
from minio import Minio


In [ ]:
plt.style.use('default')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)


In [ ]:
cwd = Path.cwd().resolve()
repo_root_candidates = [cwd, cwd.parent, Path('/home/jovyan')]
repo_root = next((candidate for candidate in repo_root_candidates if (candidate / 'config').exists()), cwd)

config_candidates = [
    repo_root / 'config' / 'asx_historic_jobs.json',
    repo_root / 'config' / 'asx_historic_jobs.test.json',
    repo_root / 'config' / 'asx_historic_jobs.example.json',
]
config_path = next((candidate for candidate in config_candidates if candidate.exists()), None)
if config_path is None:
    raise FileNotFoundError('Missing ASX jobs config. Create config/asx_historic_jobs.json or keep the test/example config in place.')

with config_path.open('r', encoding='utf-8') as handle:
    config = json.load(handle)

enabled_jobs = [job for job in config['jobs'] if job.get('enabled')]
if not enabled_jobs:
    raise ValueError(f'No enabled jobs found in {config_path.name}')

job = enabled_jobs[0]
job['config_path'] = str(config_path.relative_to(repo_root))
job


In [ ]:
def head_source_url(url: str) -> dict[str, object]:
    request = Request(url, method='HEAD', headers={'User-Agent': 'oss-data-lake-asx-notebook/1.0'})
    with urlopen(request, timeout=60) as response:
        return {
            'url': url,
            'status': getattr(response, 'status', None),
            'content_type': response.headers.get('Content-Type'),
            'content_length': response.headers.get('Content-Length'),
        }

url_status = [head_source_url(url) for url in job['source_urls']]
url_status


In [ ]:
endpoint_url = os.getenv('S3_ENDPOINT_URL', 'http://minio:9000')
parsed_endpoint = urlparse(endpoint_url)
client = Minio(
    parsed_endpoint.netloc or parsed_endpoint.path,
    access_key=os.getenv('AWS_ACCESS_KEY_ID', os.getenv('MINIO_ROOT_USER', 'minioadmin')),
    secret_key=os.getenv('AWS_SECRET_ACCESS_KEY', os.getenv('MINIO_ROOT_PASSWORD', 'minioadmin')),
    secure=parsed_endpoint.scheme == 'https',
)

raw_bucket = os.getenv('ASX_RAW_BUCKET', 'raw')
conformed_bucket = os.getenv('ASX_CONFORMED_BUCKET', 'conformed')
curated_bucket = os.getenv('ASX_CURATED_BUCKET', 'curated')
raw_prefix = job['raw_target'].strip('/')
conformed_key = job['conformed_target'].lstrip('/')
curated_key = job['curated_target'].lstrip('/')

raw_objects = list(client.list_objects(raw_bucket, prefix=raw_prefix, recursive=True))
{
    'raw_objects': [obj.object_name for obj in raw_objects],
    'conformed_exists': client.stat_object(conformed_bucket, conformed_key).size > 0,
    'curated_exists': client.stat_object(curated_bucket, curated_key).size > 0,
}


In [ ]:
def load_raw_frame(minio_client: Minio, bucket: str, object_name: str) -> tuple[pd.DataFrame, str | None]:
    response = minio_client.get_object(bucket, object_name)
    try:
        payload = response.read()
    finally:
        response.close()
        response.release_conn()

    lower_name = object_name.lower()
    if lower_name.endswith('.csv'):
        return pd.read_csv(io.BytesIO(payload)), None
    if lower_name.endswith('.xlsx'):
        workbook = pd.ExcelFile(io.BytesIO(payload), engine='openpyxl')
        sheet_name = workbook.sheet_names[0] if len(workbook.sheet_names) == 1 else str(job.get('source_options', {}).get('sheet_name', workbook.sheet_names[0]))
        return pd.read_excel(workbook, sheet_name=sheet_name, engine='openpyxl'), sheet_name
    raise ValueError(f'Unsupported raw object type: {object_name}')

if not raw_objects:
    raise FileNotFoundError('No raw objects found in MinIO. Run the DAG or ingestion step first.')

first_raw_object = raw_objects[0].object_name
raw_df, raw_sheet_name = load_raw_frame(client, raw_bucket, first_raw_object)
{'raw_object': first_raw_object, 'raw_sheet_name': raw_sheet_name, 'raw_shape': raw_df.shape}


In [ ]:
raw_df.head()


In [ ]:
raw_profile = {
    'raw_shape': raw_df.shape,
    'raw_columns': raw_df.columns.tolist(),
    'raw_null_counts': raw_df.isna().sum().to_dict(),
}
raw_profile


In [ ]:
conformed_response = client.get_object(conformed_bucket, conformed_key)
try:
    conformed_df = pd.read_parquet(io.BytesIO(conformed_response.read()))
finally:
    conformed_response.close()
    conformed_response.release_conn()

conformed_profile = {
    'conformed_shape': conformed_df.shape,
    'conformed_columns': conformed_df.columns.tolist(),
    'conformed_dtypes': {column: str(dtype) for column, dtype in conformed_df.dtypes.items()},
}
conformed_profile


In [ ]:
conformed_df.head()


## Univariate Analysis

Single-variable views for numeric distributions, categorical frequency, and null counts.


In [ ]:
null_summary = (conformed_df.isna().sum().rename('null_count').to_frame())
null_summary['null_pct'] = (null_summary['null_count'] / len(conformed_df) * 100).round(2)
null_summary.sort_values(['null_count', 'null_pct'], ascending=False)


In [ ]:
numeric_df = conformed_df.select_dtypes(include='number')
numeric_df.describe().transpose() if not numeric_df.empty else pd.DataFrame()


In [ ]:
categorical_cols = ['security_group', 'product_description_abbrev', 'source_file_type', 'source_sheet_name']
{column: conformed_df[column].fillna('Missing').astype(str).value_counts().head(10) for column in categorical_cols if column in conformed_df.columns}


In [ ]:
asx_code_frequency = (
    conformed_df['asx_code']
    .fillna('Missing')
    .astype(str)
    .value_counts()
    .rename_axis('asx_code')
    .reset_index(name='row_count')
)
asx_code_frequency


In [ ]:
plot_numeric_cols = [column for column in ['last_price'] if column in conformed_df.columns]
if plot_numeric_cols:
    fig, axes = plt.subplots(1, len(plot_numeric_cols), figsize=(7 * len(plot_numeric_cols), 4))
    axes = [axes] if len(plot_numeric_cols) == 1 else axes
    for ax, column in zip(axes, plot_numeric_cols):
        conformed_df[column].dropna().plot(kind='hist', bins=40, ax=ax, title=f'Distribution of {column}')
        ax.set_xlabel(column)
    plt.tight_layout()
else:
    print('No numeric columns available for histogram plots.')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
null_summary.sort_values('null_count', ascending=False)['null_count'].plot(kind='bar', ax=ax, title='Null counts by column')
ax.set_ylabel('Rows with null values')
plt.tight_layout()


## Bivariate Analysis

A couple of compact relationship checks between price, security groups, and missing dates.


In [ ]:
if {'security_group', 'last_price'}.issubset(conformed_df.columns):
    top_groups = conformed_df['security_group'].fillna('Missing').astype(str).value_counts().head(8).index.tolist()
    plot_df = conformed_df[conformed_df['security_group'].fillna('Missing').astype(str).isin(top_groups)].copy()
    plot_df['security_group'] = plot_df['security_group'].fillna('Missing').astype(str)
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_df.boxplot(column='last_price', by='security_group', ax=ax, rot=45)
    ax.set_title('Last price by security group')
    fig.suptitle('')
    plt.tight_layout()
else:
    print('Need security_group and last_price for boxplot analysis.')


In [ ]:
if {'last_price', 'business_date'}.issubset(conformed_df.columns):
    plot_df = conformed_df.copy()
    plot_df['business_date_missing'] = plot_df['business_date'].isna().map({True: 'Missing business_date', False: 'Present business_date'})
    fig, ax = plt.subplots(figsize=(7, 5))
    plot_df.boxplot(column='last_price', by='business_date_missing', ax=ax)
    ax.set_title('Last price by business_date availability')
    fig.suptitle('')
    plt.tight_layout()
else:
    print('Need last_price and business_date for missing-date comparison.')


## Date Coverage and Missing-Date Investigation

This section is aimed at the question about why many rows have no `business_date`.


In [ ]:
if 'business_date' in conformed_df.columns:
    business_date_non_null = conformed_df['business_date'].dropna()
    date_summary = {
        'non_null_business_date_count': int(business_date_non_null.shape[0]),
        'null_business_date_count': int(conformed_df['business_date'].isna().sum()),
        'unique_business_dates': sorted(business_date_non_null.dt.strftime('%Y-%m-%d').unique().tolist()),
        'min_business_date': business_date_non_null.min(),
        'max_business_date': business_date_non_null.max(),
    }
else:
    date_summary = 'business_date column not present in conformed data.'

date_summary


In [ ]:
if 'business_date' in conformed_df.columns:
    date_counts = conformed_df.assign(business_date=conformed_df['business_date'].dt.date).groupby('business_date', dropna=False).size().rename('row_count')
else:
    date_counts = 'business_date column not present in conformed data.'

date_counts


In [ ]:
if 'business_date' in conformed_df.columns:
    monthly_counts = conformed_df.dropna(subset=['business_date']).assign(month=lambda frame: frame['business_date'].dt.to_period('M').astype(str)).groupby('month').size().rename('row_count')
else:
    monthly_counts = 'business_date column not present in conformed data.'

monthly_counts


In [ ]:
if 'business_date' in conformed_df.columns:
    missing_date_df = conformed_df[conformed_df['business_date'].isna()].copy()
    investigation = {
        'rows_with_missing_business_date': int(len(missing_date_df)),
        'rows_with_missing_business_date_and_missing_last_price': int(missing_date_df['last_price'].isna().sum()) if 'last_price' in missing_date_df.columns else None,
        'rows_with_missing_business_date_and_present_last_price': int(missing_date_df['last_price'].notna().sum()) if 'last_price' in missing_date_df.columns else None,
    }
else:
    investigation = 'business_date column not present in conformed data.'

investigation


In [ ]:
if {'security_group', 'business_date'}.issubset(conformed_df.columns):
    missing_date_by_group = (
        conformed_df.assign(missing_business_date=conformed_df['business_date'].isna())
        .groupby('security_group', dropna=False)['missing_business_date']
        .agg(['size', 'sum'])
        .rename(columns={'size': 'row_count', 'sum': 'missing_business_date_count'})
        .sort_values(['missing_business_date_count', 'row_count'], ascending=False)
        .head(15)
    )
else:
    missing_date_by_group = 'Need security_group and business_date to inspect missing-date concentration.'

missing_date_by_group


In [ ]:
if {'product_description_abbrev', 'business_date'}.issubset(conformed_df.columns):
    missing_date_by_product = (
        conformed_df.assign(missing_business_date=conformed_df['business_date'].isna())
        .groupby('product_description_abbrev', dropna=False)['missing_business_date']
        .agg(['size', 'sum'])
        .rename(columns={'size': 'row_count', 'sum': 'missing_business_date_count'})
        .sort_values(['missing_business_date_count', 'row_count'], ascending=False)
        .head(15)
    )
else:
    missing_date_by_product = 'Need product_description_abbrev and business_date to inspect missing-date concentration.'

missing_date_by_product


In [ ]:
if {'security_group', 'business_date'}.issubset(conformed_df.columns):
    plot_df = conformed_df.assign(missing_business_date=conformed_df['business_date'].isna())
    top_groups = plot_df['security_group'].fillna('Missing').astype(str).value_counts().head(10).index.tolist()
    grouped = (
        plot_df[plot_df['security_group'].fillna('Missing').astype(str).isin(top_groups)]
        .assign(security_group=lambda frame: frame['security_group'].fillna('Missing').astype(str))
        .groupby(['security_group', 'missing_business_date'])
        .size()
        .unstack(fill_value=0)
    )
    fig, ax = plt.subplots(figsize=(11, 5))
    grouped.rename(columns={False: 'Present', True: 'Missing'}).plot(kind='bar', stacked=True, ax=ax, title='business_date presence by security group')
    ax.set_ylabel('Rows')
    plt.tight_layout()
else:
    print('Need security_group and business_date for stacked missing-date plot.')


In [ ]:
curated_response = client.get_object(curated_bucket, curated_key)
try:
    curated_summary = json.loads(curated_response.read().decode('utf-8'))
finally:
    curated_response.close()
    curated_response.release_conn()

local_curated_path = repo_root / 'data' / 'curated' / curated_key
curated_summary['local_curated_path'] = str(local_curated_path) if local_curated_path.exists() else None
curated_summary
